## Volume Drop Alert
Monitors daily data volume and fires an alert when it drops.

- **Metric:** `volume_ratio` = rows on the latest day / average rows over the previous 7 days
- **Alert condition:** `volume_ratio < 0.5` (volume dropped by more than half)
- **Simulation:** done on a sandbox copy of the fact table, so production gold

In [0]:
%run ./00_config

In [0]:
sandbox_table = f"{catalog}.{gold_schema}.fact_volume_sandbox"

spark.sql(f"""
    CREATE OR REPLACE TABLE {sandbox_table}
    COMMENT 'Sandbox copy of fact_video_daily_stats (real period only) used to simulate a volume drop for the alert demo. Safe to drop.'
    AS
    SELECT *
    FROM {gold['fact_video_daily_stats']}
    WHERE date_key <= 20180614
""")

# display(spark.sql(f"SELECT COUNT(*) AS rows_cnt FROM {sandbox_table}"))

In [0]:
volume_check_sql = f"""
    WITH daily AS (
        SELECT date_key, COUNT(*) AS rows_cnt
        FROM {sandbox_table}
        GROUP BY date_key
    ),
    ranked AS (
        SELECT date_key,
               rows_cnt,
               ROW_NUMBER() OVER (ORDER BY date_key DESC) AS day_rank
        FROM daily
    )
    SELECT
        MAX(CASE WHEN day_rank = 1 THEN date_key END) AS latest_date_key,
        MAX(CASE WHEN day_rank = 1 THEN rows_cnt END) AS latest_rows,
        ROUND(AVG(CASE WHEN day_rank BETWEEN 2 AND 8 THEN rows_cnt END), 1) AS avg_prev_7_days,
        ROUND(
            MAX(CASE WHEN day_rank = 1 THEN rows_cnt END) * 100
            / AVG(CASE WHEN day_rank BETWEEN 2 AND 8 THEN rows_cnt END), 3) AS volume_pct
    FROM ranked
"""

# display(spark.sql(volume_check_sql))

### Simulate a volume drop
Insert a new day (2018-06-15) with only 5 rows, while a normal day has ~200.

In [0]:
spark.sql(f"""
    INSERT INTO {sandbox_table}
        (date_key, video_id, channel_key, category_id,
         views, likes, dislikes, comment_count,
         views_delta, trending_day_num, _updated_at)
    SELECT
        20180615 AS date_key,
        video_id, channel_key, category_id,
        views, likes, dislikes, comment_count,
        views_delta, trending_day_num,
        current_timestamp() AS _updated_at
    FROM {sandbox_table}
    WHERE date_key = 20180614
    LIMIT 5
""")

# display(spark.sql(volume_check_sql))

In [0]:
spark.sql(f"DELETE FROM {sandbox_table} WHERE date_key = 20180615")

# display(spark.sql(volume_check_sql))